# Homework 3 — Machine Learning Classification

## Homework Objective

The objective of this homework is to build and evaluate a **logistic regression model** for predicting whether a lead converts (signs up to the platform).

The notebook covers data preparation, missing-value handling, exploratory analysis, train/validation/test splitting, mutual information, one-hot encoding, feature elimination, and regularization.

**Target:** `converted`  
**Features:** `lead_source`, `industry`, `employment_status`, `location`, `annual_income`, `number_of_courses_viewed`, `interaction_count`, `lead_score`


# Cohort Homework-03 classification model

In [17]:
# Importing modules
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import mutual_info_score, accuracy_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression

## Loading data

In [18]:
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv"
df = pd.read_csv(url)

df.head()

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score,converted
0,organic_search,technology,employed,europe,88160.0,3,4,0.64,1
1,social_media,technology,employed,south_america,72688.0,3,7,0.72,1
2,referral,retail,employed,europe,44697.0,3,4,0.58,1
3,paid_ads,manufacturing,student,NaN,NaN,3,6,0.57,0
4,referral,manufacturing,student,north_america,24062.0,4,5,0.62,1


Understanding the data set and cleaning

In [19]:
df.shape

(5000, 9)

In [20]:
df.dtypes

lead_source                     str
industry                        str
employment_status               str
location                        str
annual_income               float64
number_of_courses_viewed      int64
interaction_count             int64
lead_score                  float64
converted                     int64
dtype: object

In [21]:
df.isnull().sum()

lead_source                 148
industry                    240
employment_status           194
location                    208
annual_income               369
number_of_courses_viewed      0
interaction_count             0
lead_score                   35
converted                     0
dtype: int64

In [22]:
cat_cols = df.select_dtypes(include='str').columns.to_list()
num_cols = df.select_dtypes(exclude='str').columns.to_list()

In [23]:
cat_cols

['lead_source', 'industry', 'employment_status', 'location']

In [24]:
num_cols

['annual_income',
 'number_of_courses_viewed',
 'interaction_count',
 'lead_score',
 'converted']

Handling missing values: categorical features are filled with `'NA'`, numerical features with `0.0`

In [25]:
df[cat_cols] = df[cat_cols].fillna('NA')
df[num_cols] = df[num_cols].fillna(0.0)

df.isnull().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

# EDA

Question 1: most frequent value (mode) of `industry`

In [26]:
df.industry.value_counts()

industry
technology       1173
retail            925
healthcare        840
finance           720
education         639
manufacturing     463
NA                240
Name: count, dtype: int64

Question 2: correlation matrix of the numerical features (target excluded)

In [27]:
corr = df[num_cols].drop(columns='converted').corr()
corr

,annual_income,number_of_courses_viewed,interaction_count,lead_score
annual_income,1.000000,0.161300,0.122842,0.229496
number_of_courses_viewed,0.161300,1.000000,0.721609,0.757204
interaction_count,0.122842,0.721609,1.000000,0.915746
lead_score,0.229496,0.757204,0.915746,1.000000


# Train test splits

In [28]:
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=42)
df_train, df_val = train_test_split(df_full_train, test_size=0.25, random_state=42)

Target separation

In [29]:
y_train = df_train.converted.values
y_val = df_val.converted.values
y_test = df_test.converted.values

del df_train['converted']
del df_val['converted']
del df_test['converted']

print(df_train.shape, df_val.shape, df_test.shape)

(3000, 8) (1000, 8) (1000, 8)


# Mutual information

Question 3: mutual information between `converted` and each categorical variable (training set only)

In [30]:
for col in cat_cols:
    score = mutual_info_score(df_train[col], y_train)
    print(col, round(score, 2))

lead_source 0.03
industry 0.0
employment_status 0.02
location 0.0


# Training Logistic Regression model

Question 4: one-hot encoding with `DictVectorizer`, accuracy on the validation set

In [31]:
def train_and_eval(features, C=1.0):
    dv = DictVectorizer(sparse=False)
    X_train = dv.fit_transform(df_train[features].to_dict(orient='records'))
    X_val = dv.transform(df_val[features].to_dict(orient='records'))

    model = LogisticRegression(solver='liblinear', C=C, max_iter=1000, random_state=42)
    model.fit(X_train, y_train)
    return accuracy_score(y_val, model.predict(X_val))

all_features = cat_cols + [c for c in num_cols if c != 'converted']

In [32]:
acc = train_and_eval(all_features)
print(acc, round(acc, 2))

0.645 0.65


Baseline check: fraction of converted leads in the validation set (accuracy of always predicting the majority class)

In [33]:
y_val.mean()

np.float64(0.597)

# Feature elimination

Question 5: accuracy difference after dropping each feature (baseline is the unrounded Q4 accuracy)

In [34]:
for f in ['lead_source', 'number_of_courses_viewed', 'interaction_count']:
    feats = [x for x in all_features if x != f]
    print(f, acc - train_and_eval(feats))

lead_source 0.0030000000000000027
number_of_courses_viewed 0.0020000000000000018
interaction_count 0.04400000000000004


# Regularization

Question 6: regularized logistic regression, accuracy on the validation set for different values of `C`

In [35]:
for C in [0.000001, 0.00001, 0.0001, 0.001]:
    print(C, round(train_and_eval(all_features, C=C), 3))

1e-06 0.598
1e-05 0.598
0.0001 0.613
0.001 0.645


## Final Answers

### Question 1
ans: **`technology`**

### Question 2
ans: **`interaction_count` and `lead_score`**

### Question 3
ans: **`lead_source`**

### Question 4
ans: Validation accuracy **`0.65`**

### Question 5
ans: Smallest difference: **`number_of_courses_viewed`**

### Question 6
ans: Best regularization parameter: **`C = 0.001`**
